In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras import layers, models, regularizers
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

In [2]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print(x_train.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 14s 0us/step
(50000, 32, 32, 3)


In [3]:
IMG_SIZE = 224
BATCH_SIZE = 16   # small to avoid GPU memory issues
NUM_CLASSES = 10

In [4]:
def preprocess(image, label):
    image = tf.image.resize(image, (224,224))
    image = preprocess_input(image)
    return image, label

In [5]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1)
])

In [6]:
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(10000)
train_ds = train_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_ds = test_ds.map(preprocess)
test_ds = test_ds.batch(BATCH_SIZE)

In [7]:
base_model = VGG16(
    include_top=False,
    weights='imagenet',
    input_shape=(224,224,3)
)

base_model.trainable = False

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [8]:
inputs = tf.keras.Input(shape=(224,224,3))

x = data_augmentation(inputs)
x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(
    512,
    kernel_initializer='he_normal',
    kernel_regularizer=regularizers.l2(1e-4)
)(x)

x = layers.BatchNormalization()(x)
x = layers.ReLU()(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs)

In [9]:
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [10]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3
    ),

    tf.keras.callbacks.ModelCheckpoint(
        "best_vgg16_cifar10.h5",
        save_best_only=True
    )
]

In [ ]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.6463 - loss: 1.1430

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 279s 86ms/step - accuracy: 0.6463 - loss: 1.1429 - val_accuracy: 0.8309 - val_loss: 0.6066 - learning_rate: 0.0010
Epoch 2/20
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 272s 87ms/step - accuracy: 0.7503 - loss: 0.8439 - val_accuracy: 0.8293 - val_loss: 0.6199 - learning_rate: 0.0010
Epoch 3/20
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.7613 - loss: 0.8211

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 273s 87ms/step - accuracy: 0.7613 - loss: 0.8211 - val_accuracy: 0.8440 - val_loss: 0.5910 - learning_rate: 0.0010
Epoch 4/20
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 274s 88ms/step - accuracy: 0.7705 - loss: 0.8014 - val_accuracy: 0.8442 - val_loss: 0.5921 - learning_rate: 0.0010
Epoch 5/20
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 322s 88ms/step - accuracy: 0.7727 - loss: 0.7985 - val_accuracy: 0.8433 - val_loss: 0.6022 - learning_rate: 0.0010
Epoch 6/20
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.7742 - loss: 0.7953

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 274s 88ms/step - accuracy: 0.7742 - loss: 0.7953 - val_accuracy: 0.8473 - val_loss: 0.5772 - learning_rate: 0.0010
Epoch 7/20
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 274s 88ms/step - accuracy: 0.7772 - loss: 0.7924 - val_accuracy: 0.8470 - val_loss: 0.5840 - learning_rate: 0.0010
Epoch 8/20
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 274s 88ms/step - accuracy: 0.7795 - loss: 0.7858 - val_accuracy: 0.8526 - val_loss: 0.5802 - learning_rate: 0.0010
Epoch 9/20
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 323s 88ms/step - accuracy: 0.7798 - loss: 0.7878 - val_accuracy: 0.8392 - val_loss: 0.6111 - learning_rate: 0.0010
Epoch 10/20
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.7874 - loss: 0.7540

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 322s 88ms/step - accuracy: 0.7874 - loss: 0.7540 - val_accuracy: 0.8632 - val_loss: 0.5371 - learning_rate: 2.0000e-04
Epoch 11/20
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.8020 - loss: 0.7074

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 275s 88ms/step - accuracy: 0.8020 - loss: 0.7074 - val_accuracy: 0.8632 - val_loss: 0.5169 - learning_rate: 2.0000e-04
Epoch 12/20
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.8067 - loss: 0.6790

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 275s 88ms/step - accuracy: 0.8067 - loss: 0.6790 - val_accuracy: 0.8616 - val_loss: 0.5075 - learning_rate: 2.0000e-04
Epoch 13/20
2222/3125 ━━━━━━━━━━━━━━━━━━━━ 1:06 74ms/step - accuracy: 0.8045 - loss: 0.6760